In [2]:
import requests
import json
import os

USERNAME = "darshanijadhav29@gmail.com"
PASSWORD = "_Darshanijadhav29_"

SAVE_PATH = "/home/darshani/lightkurve-env/space-debris-detector/data/raw/TLE_latestt.json"

LOGIN_URL = "https://www.space-track.org/ajaxauth/login"

GP_URL = (
    "https://www.space-track.org/basicspacedata/query/"
    "class/gp/"
    "orderby/epoch desc/"
    "limit/500000/"
    "format/json")

session = requests.Session()

login_payload = {
    "identity": USERNAME,
    "password": PASSWORD
}

print("Logging in to Space-Track...")

response = session.post(LOGIN_URL, data=login_payload)

if response.status_code != 200:
    raise Exception("Login failed! Check username/password.")

print("Login successful.")


print("Fetching GP (TLE) latest data...")

response = session.get(GP_URL)

if response.status_code != 200:
    raise Exception(f"GP fetch failed: {response.text}")

gp_data = response.json()
print(f"Fetched {len(gp_data)} GP records.")

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

with open(SAVE_PATH, "w") as f:
    json.dump(gp_data, f, indent=2)

print(f"Saved GP latest data to: {SAVE_PATH}")


Logging in to Space-Track...
Login successful.
Fetching GP (TLE) latest data...
Fetched 66666 GP records.
Saved GP latest data to: /home/darshani/lightkurve-env/space-debris-detector/data/raw/TLE_latestt.json


In [2]:
import requests
import time

TOKEN = "IjYwMzY2YWE0LTdkMTYtNDYyOC04N2JhLWI5YjUxMGYwMjEyZCI.cNklPBs1zYFgKpd0To_ZV659x1I"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.api+json"
}

all_objects = []
page_number = 1
page_size = 100

while True:
    url = f"https://discosweb.esoc.esa.int/api/objects?page[number]={page_number}&page[size]={page_size}"
    
    # Retry loop
    for attempt in range(5):  # try up to 5 times
        try:
            response = requests.get(url, headers=headers, timeout=20)
            
            if response.status_code == 503:
                print(f"Page {page_number}: 503, retrying in 5 sec...")
                time.sleep(5)
                continue
            
            if response.status_code != 200:
                print("Error:", response.text)
                raise Exception("Failed page.")
            
            break  # success, exit retry loop
        
        except Exception as e:
            print(f"Attempt {attempt+1} failed. Waiting 5 sec...")
            time.sleep(5)
    else:
        print(f"Page {page_number} failed after 5 retries. Stopping.")
        break

    data = response.json()
    objects = data.get("data", [])

    if not objects:
        print("No more data.")
        break

    all_objects.extend(objects)
    print(f"Fetched page {page_number}, total={len(all_objects)}")

    next_link = data.get("links", {}).get("next")
    if not next_link:
        print("Reached last page.")
        break

    page_number += 1
    time.sleep(1)  # prevent rate limit

print(f"Final total objects: {len(all_objects)}")

Fetched page 1, total=100
Fetched page 2, total=200
Fetched page 3, total=300
Fetched page 4, total=400
Fetched page 5, total=500
Fetched page 6, total=600
Fetched page 7, total=700
Fetched page 8, total=800
Fetched page 9, total=900
Fetched page 10, total=1000
Fetched page 11, total=1100
Fetched page 12, total=1200
Fetched page 13, total=1300
Attempt 1 failed. Waiting 5 sec...
Fetched page 14, total=1400
Fetched page 15, total=1500
Fetched page 16, total=1600
Fetched page 17, total=1700
Fetched page 18, total=1800
Attempt 1 failed. Waiting 5 sec...
Fetched page 19, total=1900
Fetched page 20, total=2000
Fetched page 21, total=2100
Fetched page 22, total=2200
Attempt 1 failed. Waiting 5 sec...
Fetched page 23, total=2300
Fetched page 24, total=2400
Fetched page 25, total=2500
Fetched page 26, total=2600
Fetched page 27, total=2700
Fetched page 28, total=2800
Attempt 1 failed. Waiting 5 sec...
Fetched page 29, total=2900
Fetched page 30, total=3000
Fetched page 31, total=3100
Fetched pa

In [ ]:
import os
import json

output_path = os.path.expanduser("/home/darshani/lightkurve-env/space-debris-detector/data/raw/discos_objects_latest.json")

# Create the parent directory
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Now save the file
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_objects, f, indent=4)

print(f"Saved to {output_path}")


Saved to /home/darshani/lightkurve-env/space-debris-detector/data/raw/discoss_objects.json


In [ ]:
#!/usr/bin/env python3
"""
Download ALL Space-Track.org CDM (Conjunction Data Messages) in JSON format
Saves to: /home/darshani/lightkurve-env/space-debris-detector/data/raw/
"""

import requests
import json
import os
import pandas as pd
from datetime import datetime
import sys

# === CONFIGURATION ===
USERNAME = 'darshanijadhav29@gmail.com'  # Replace with your Space-Track.org username
PASSWORD = '_Darshanijadhav29_'  # Replace with your Space-Track.org password
SAVE_DIR = '/home/darshani/lightkurve-env/space-debris-detector/data/raw'
OUTPUT_JSON = 'conjunction_cdm.json'
OUTPUT_CSV = 'all_conjunction_cdm.csv'

def ensure_directory(directory):
    """Create directory if it doesn't exist"""
    os.makedirs(directory, exist_ok=True)
    print(f"✓ Directory ready: {directory}")

def login_to_spacetrack(username, password):
    """Login to Space-Track.org and return authenticated session"""
    print("🔐 Logging into Space-Track.org...")
    
    session = requests.Session()
    login_data = {
        'identity': username,
        'password': password
    }
    
    login_url = 'https://www.space-track.org/ajaxauth/login'
    login_response = session.post(login_url, data=login_data)
    
    if 'Login: Failed' in login_response.text:
        raise ValueError("❌ LOGIN FAILED: Check your username/password")
    
    print("✓ Login successful!")
    return session

def download_cdm_data(session):
    """Download ALL Conjunction Data Messages in JSON format"""
    print("📥 Downloading ALL CDM conjunction predictions...")
    
    # Query ALL CDMs (no filters = complete conjunction table)
    # CDM endpoint shows active conjunctions (close approaches)
    url = 'https://www.space-track.org/basicspacedata/query/class/cdm/format/json'
    
    response = session.get(url)
    response.raise_for_status()  # Raises exception for HTTP errors
    
    cdm_data = response.json()
    print(f"✓ Downloaded {len(cdm_data)} conjunction events")
    
    return cdm_data

def save_data(data, save_dir):
    """Save JSON and CSV files with timestamp"""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # JSON file
    json_path = os.path.join(save_dir, f"{OUTPUT_JSON.replace('.json', f'_{timestamp}.json')}")
    with open(json_path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"✓ JSON saved: {json_path}")
    
    # CSV file (flattened)
    df = pd.json_normalize(data)
    csv_path = os.path.join(save_dir, f"{OUTPUT_CSV.replace('.csv', f'_{timestamp}.csv')}")
    df.to_csv(csv_path, index=False)
    print(f"✓ CSV saved: {csv_path}")
    
    return json_path, csv_path

def main():
    """Main execution function with full error handling"""
    try:
        # 1. Check required libraries
        required_libs = ['requests', 'pandas']
        for lib in required_libs:
            __import__(lib)
        
        # 2. Validate credentials
        if USERNAME == 'your_username_here' or PASSWORD == 'your_password_here':
            raise ValueError("❌ Please set your Space-Track.org username/password in USERNAME/PASSWORD variables")
        
        # 3. Setup directory
        ensure_directory(SAVE_DIR)
        
        # 4. Login
        session = login_to_spacetrack(USERNAME, PASSWORD)
        
        # 5. Download CDM data
        cdm_data = download_cdm_data(session)
        
        # 6. Logout
        session.get('https://www.space-track.org/ajaxauth/logout')
        print("✓ Logged out")
        
        # 7. Save files
        json_path, csv_path = save_data(cdm_data, SAVE_DIR)
        
        # 8. Show preview
        df = pd.read_csv(csv_path)
        key_cols = ['OBJECT1', 'OBJECT2', 'TCA_UTC', 'MAX_PC'] if 'OBJECT1' in df.columns else df.columns[:4].tolist()
        print("\n📊 PREVIEW (first 5 conjunctions):")
        print(df[key_cols].head())
        
        print(f"\n🎉 SUCCESS! CDM files saved to {SAVE_DIR}")
        print(f"   JSON: {json_path}")
        print(f"   CSV:  {csv_path}")
        
    except ImportError as e:
        print(f"❌ Missing library: pip install requests pandas")
        sys.exit(1)
    except ValueError as e:
        print(f"❌ {e}")
        sys.exit(1)
    except requests.exceptions.RequestException as e:
        print(f"❌ Network/API error: {e}")
        print("💡 CDM endpoint limits: Check daily query limits")
        sys.exit(1)
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()


In [4]:
import requests
import pandas as pd

USERNAME = "darshanijadhav29@gmail.com"
PASSWORD = "_Darshanijadhav29_"

LOGIN_URL = "https://www.space-track.org/ajaxauth/login"
DECAY_URL = (
    "https://www.space-track.org/basicspacedata/query/"
    "class/decay/format/json"
)

session = requests.Session()

# 1. Login
login_data = {
    "identity": USERNAME,
    "password": PASSWORD
}

print("Logging in...")
resp = session.post(LOGIN_URL, data=login_data)

if resp.status_code != 200:
    raise Exception(f"Login failed: {resp.status_code} {resp.text}")

print("Login successful.")

# 2. Get Decay Data
print("Fetching decay data...")
decay_resp = session.get(DECAY_URL)

if decay_resp.status_code != 200:
    raise Exception("Error fetching decay data.")

decay_json = decay_resp.json()

# Convert to DataFrame
df = pd.DataFrame(decay_json)

print(df.head())
df.to_json("decay_data.json", index=False)
print("Saved to decay_data.json")

output_path = "/home/darshani/lightkurve-env/space-debris-detector/data/raw/decay_latest.json"

with open(output_path, "w") as f:
    json.dump(decay_json, f, indent=4)

print("Saved JSON:", output_path)

Logging in...
Login successful.
Fetching decay data...
  NORAD_CAT_ID OBJECT_NUMBER OBJECT_NAME    INTLDES  OBJECT_ID RCS RCS_SIZE  \
0            1             1    SL-1 R/B  1957-001A  1957-001A   0    LARGE   
1            4             4  EXPLORER 1  1958-001A  1958-001A   0     None   
2            7             7    SL-1 R/B  1958-004A  1958-004A   0     None   
3            8             8   SPUTNIK 3  1958-004B  1958-004B   0    LARGE   
4            9             9  EXPLORER 4  1958-005A  1958-005A   0     None   

  COUNTRY            MSG_EPOCH         DECAY_EPOCH  SOURCE    MSG_TYPE  \
0     CIS  1957-12-01 00:00:00  1957-12-01 0:00:00  satcat  Historical   
1      US  1970-03-31 00:00:00  1970-03-31 0:00:00  satcat  Historical   
2     CIS  1958-12-03 00:00:00  1958-12-03 0:00:00  satcat  Historical   
3     CIS  1960-04-06 00:00:00  1960-04-06 0:00:00  satcat  Historical   
4      US  1959-10-23 00:00:00  1959-10-23 0:00:00  satcat  Historical   

  PRECEDENCE  
0         

In [ ]:
import json
import os

output_path = "/home/darshani/lightkurve-env/space-debris-detector/.gitignore/data/raw/decay_latest.json"

with open(output_path, "w") as f:
    json.dump(decay_json, f, indent=4)

print("Saved JSON:", output_path)


Saved JSON: /home/darshani/lightkurve-env/space-debris-detector/data/raw/decay.json


In [5]:
import requests
import json
import os

USERNAME = "darshanijadhav29@gmail.com"
PASSWORD = "_Darshanijadhav29_"

LOGIN_URL = "https://www.space-track.org/ajaxauth/login"
CDM_URL = "https://www.space-track.org/basicspacedata/query/class/cdm_public/format/json"

SAVE_PATH = "/home/darshani/lightkurve-env/space-debris-detector/data/raw/conjunctions_latest.json"

def login_and_download():
    session = requests.Session()

    print("Logging in...")
    response = session.post(LOGIN_URL, data={"identity": USERNAME, "password": PASSWORD})

    print("Status:", response.status_code)
    print("Text:", response.text)
    print("Cookies:", session.cookies.get_dict())

    if response.status_code != 200:
        raise Exception("Login failed")

    print("Downloading CDM Public data...")
    r = session.get(CDM_URL)

    if r.status_code != 200:
        raise Exception(f"Download failed: {r.status_code}")

    data = r.json()

    # Ensure directory exists
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

    # Write JSON safely (pretty formatted)
    with open(SAVE_PATH, "w") as f:
        json.dump(data, f, indent=2)

    print(f"Saved {len(data)} CDM records to:")
    print(SAVE_PATH)

login_and_download()


Logging in...
Status: 200
Text: ""
Cookies: {'chocolatechip': '21orvrr8j3egd47bdhlfohgd5tfrcfgj'}
Saved 3945 CDM records to:
/home/darshani/lightkurve-env/space-debris-detector/data/raw/conjunctions_latest.json


In [ ]:
import requests
import json
import os

USERNAME = "darshanijadhav29@gmail.com"
PASSWORD = "_Darshanijadhav29_"

SAVE_PATH = "/home/darshani/lightkurve-env/space-debris-detector/data/raw/boxscore_latest.json"

LOGIN_URL = "https://www.space-track.org/ajaxauth/login"
BOX_URL = "https://www.space-track.org/basicspacedata/query/class/boxscore/format/json"

session = requests.Session()

print("Logging in...")
login_data = {"identity": USERNAME, "password": PASSWORD}
resp = session.post(LOGIN_URL, data=login_data)

print("Status:", resp.status_code)
print("Text:", resp.text)
print("Cookies:", session.cookies.get_dict())

if "chocolatechip" not in session.cookies.get_dict():
    raise Exception("Login failed.")

print("Login successful.")
print("----------------------------------------")
print("Downloading Boxscore data...")

resp = session.get(BOX_URL)

if resp.status_code != 200:
    raise Exception(f"Boxscore request failed: {resp.status_code}\n{resp.text}")

data = resp.json()

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

with open(SAVE_PATH, "w") as f:
    json.dump(data, f, indent=4)

print(f"Saved {len(data)} boxscore records to:\n{SAVE_PATH}")


Logging in...
Status: 200
Text: ""
Cookies: {'chocolatechip': 'q1lim054bpbjp1hbhj2s0rvv8npcpu3i'}
Login successful.
----------------------------------------
Saved 122 boxscore records to:
/home/darshani/lightkurve-env/space-debris-detector/data/raw/boxscore.json


In [14]:
import pandas as pd
import json
from pathlib import Path

RAW = Path("data/raw")
PROCESSED = Path("data/processed")

PROCESSED.mkdir(parents=True, exist_ok=True)

files = {
    "tle": "tle.json",
    "satcat": "satcat.json",
    "discos": "discos_objects.json",
    "conjunctions": "conjunctions.json",
    "decay": "decay.json",
    "boxscore": "boxscore.json"
}

def clean_and_save(input_name, output_name):
    raw_path = RAW / input_name
    df = pd.read_json(raw_path)
    
    # Basic cleaning
    df = df.replace("", None)
    df = df.drop_duplicates()
    
    # Save Parquet
    df.to_parquet(PROCESSED / output_name, index=False)
    print(f"Saved: {output_name} ({len(df)} rows)")

for key, file in files.items():
    clean_and_save(file, f"{key}_cleaned.parquet")


FileNotFoundError: File data/raw/tle.json does not exist